# 1) About

## Purpose
This notebook transforms raw transaction data into a **temporal behavioral dataset** suitable for Machine Learning Modeling. Each row in final dataset represents one client at one monthly snapshot, with aggregated behavioral features and a **forward looking** fraud label.


## Connection to EDA
In the EDA Notebook, we established:

- Dataset contains 2 AML patterns. Fan-in (multiple senders -> one receiver) and Cycle (circular flows).
- Only ~0.13% of the transactions are fraudulent -> dataset heavily imbalanced.
- Fraudulent clients demonstrate distinct behavioral signals -> 2x unique senders/ 2x received trx volume, ~60% more transactions.
- Transaction network is sparse (1.3% density), which makes 'counterparty count' related features discriminative.

## Outcome of the NB
This notebook will transform raw csv files (accounts/transactions/alerts) into a client-month level dataset ready for train-test split.

## Feature Categories
1. Behavioral Intensity: Trx counts/volumes
2. Fan-in Signals: Unique senders, sender diversity, received trx volume
3. Cycle Signals: Sender-receiver overlap, bidirectional behavior patterns
4. Velocity/Change: Activity spikes, dormancy-activity transitions, new counterparties.

# 2) Imports and Data Loading

In [16]:
import pandas as pd, numpy as np 
from pathlib import Path 
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# import raw data
data_path = Path("../data/raw")

dfs = {}
for f in data_path.glob("*.csv"):
    dfs[f.stem] = pd.read_csv(f)


# handling timestamp - assuming 2020-01-01 as the start date of the timestamp
start_date = pd.to_datetime("2020-01-01")
for i, df in dfs.items():
    if 'TIMESTAMP' in df.columns:
        df['date'] = start_date + pd.to_timedelta(df['TIMESTAMP'], unit='D')

df_accounts = dfs['accounts']
df_transactions = dfs['transactions']
df_alerts = dfs['alerts']   

# Tagging fraudulent accounts
fraudulent_accounts = list(df_accounts[df_accounts['IS_FRAUD'] == 1]['ACCOUNT_ID'].unique())

# General Overview
print(f"Accounts: {len(df_accounts):,}, Fraudulent accounts: {len(fraudulent_accounts)}")
print(f"All Transactions: {len(df_transactions):,}, Alerted Transactions: {len(df_alerts):,}")
print(f"Date range of trx: {df_transactions['date'].min().date()} to {df_transactions['date'].max().date()}")


Accounts: 10,000, Fraudulent accounts: 1685
All Transactions: 1,323,234, Alerted Transactions: 1,719
Date range of trx: 2020-01-01 to 2020-07-18


# 3) Snapshot Calendar

Define monthly snapshot dates. Each one represents a "prediction point" where we ask: **will this client exhibit suspicious behavior in the upcoming month?**

**Design rules:**
- Features use **only past data** (up to snapshot date-exclusive)
- Labels look at the **full calendar month ahead** (snapshot month)
- Need at least **3 full calendar months of history** for rolling features
- Trx data range: 2020-01-01 to 2020-07-18

This means:
- **First usable snapshot:** April 1 (Jan, Feb, Mar -> 3 full months of history)
- **Last usable snapshot:** June 1 (needs full June for labels; July only has 18 days)

In [17]:
data_start = df_transactions['date'].min()
data_end = df_transactions['date'].max()
print(f"Data covers a period of {(data_end - data_start).days} days")
print(f"Date Range of Transactions: {data_start} to {data_end}")

# Create Monthly Snapshots

# min history - 3 full calendar months
min_history_months = 3

# all month starts
month_starts = pd.date_range(start=data_start, end=data_end, freq = "MS")

# snapshot_dates -> 3 calendar months of history, 1 calendar month of future
snapshot_dates = []
for i in month_starts:
    # does it have enough future data --1M
    month_end = i + pd.offsets.MonthEnd(0)
    has_full_month_ahead = month_end <= data_end

    # does it have enough history --3M 
    history_start = i - pd.DateOffset(months = min_history_months)
    has_enough_history = history_start >= data_start

    if has_full_month_ahead and has_enough_history:
        snapshot_dates.append(i)

print(f"\nUsable snapshot dates ({len(snapshot_dates)}):")

# print each snapshot date's rolling window history start and prediction date
for i in snapshot_dates:
    history_start = i - pd.DateOffset(months = min_history_months) 
    history_end =  i - pd.Timedelta(days=1)
    prediction_end = i + pd.offsets.MonthEnd(0)
        # print(f"SnapshotDate {i}, uses history from {history_start} to i - , {month_end}")
    print(f" {i.date()}  |  history from: {history_start.date()} to {history_end.date()} |  predicts: {i.date()} until {prediction_end.date()}")


Data covers a period of 199 days
Date Range of Transactions: 2020-01-01 00:00:00 to 2020-07-18 00:00:00

Usable snapshot dates (3):
 2020-04-01  |  history from: 2020-01-01 to 2020-03-31 |  predicts: 2020-04-01 until 2020-04-30
 2020-05-01  |  history from: 2020-02-01 to 2020-04-30 |  predicts: 2020-05-01 until 2020-05-31
 2020-06-01  |  history from: 2020-03-01 to 2020-05-31 |  predicts: 2020-06-01 until 2020-06-30


# 4) Feature Engineering

For each client at each snapshot date, we will compute behavioral futures using transactions that occured before the snapshot date.  



## 4.a Behavioral Intensity

General transaction behavior per client, computed for both **Recent** (previous 1 month) and **Baseline** (2 months before that) windows. 

Windows are non-overlapping so velocity ratios give clean change signals.

Features per window:
- Transaction counts, volumes, and sizes (sent/received)

In [18]:
### This function is more optimized, it filters the data by window first, then filters by account

def compute_intensity_features(df_transactions, snapshot, window_start, prefix):
    """
    Compute behavioral intensity features for all accounts within a time window - 
    prefix: last 1 month or 2 months before last month
    Returns a df with one row per account
    """

    # Filter trx per window of interest (last month or 2 months before last month)
    window_trx = df_transactions[(df_transactions["date"] >= window_start) & (df_transactions["date"] < snapshot)]

    # 1. Sending behavior per client
    sent = (
        df_transactions.groupby("SENDER_ACCOUNT_ID")
        .agg(
            trx_sent=("TX_ID", "nunique"),
            amt_sent=("TX_AMOUNT", "sum"),
            avg_amt_sent=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # 2. Receiving behavior per client
    received = (
        df_transactions.groupby("RECEIVER_ACCOUNT_ID")
        .agg(
            trx_received=("TX_ID", "nunique"),
            amt_received=("TX_AMOUNT", "sum"),
            avg_amt_received=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # Combine features
    features = sent.join(received, "ACCOUNT_ID", "outer").fillna(0)
    # #rename cols
    features.columns =[f"{prefix}_{col}" for col in features.columns]
    return features

# Test: compute recent features for April 1 snapshot
test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

recent_features = compute_intensity_features(df_transactions, test_snapshot, test_recent_start, "recent")
print(f"Snapshot: {test_snapshot.date()}")
print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"Accounts with activity: {len(recent_features)}")
print(f"\nSample (first 5 accounts):")
recent_features.head()


Snapshot: 2020-04-01
Window: 2020-03-01 to 2020-04-01
Accounts with activity: 9999

Sample (first 5 accounts):


,recent_trx_sent,recent_amt_sent,recent_avg_amt_sent,recent_trx_received,recent_amt_received,recent_avg_amt_received
ACCOUNT_ID,,,,,,
1,24,4219.20,175.80,0.0,0.00,0.00
2,18,2556.90,142.05,0.0,0.00,0.00
3,19,2391.72,125.88,23.0,313.49,13.63
4,23,3475.99,151.13,0.0,0.00,0.00
5,38,2669.12,70.24,19.0,156.75,8.25


## 4.b Fan-In Signals

Features that capture fan-in signals -> multiple senders funneling money to a single receiver. 

From the EDA, we know that fraudulent clients have 2x as many unique senders and 2x received volume compared to non-fraud clients.

Features:

- Unique Senders: Number of unique accounts sending funds to this account
- Received Volume Concentration: What % of received trx comes from the top sender (A low concentration of MANY senders is suspicious)
- Sender Diversity Ratio: Unique senders / total received trx (lots of different counterparties sending money is suspicious)
- Received-to-send ratio: Amount received/ amount send (receiving more way more than sending - fan-in indicator)

In [19]:
def compute_fanin_features(df_transactions, snapshot, window_start, prefix):
    """
    Compute fan-in features for ALL accounts within a time window.
    """
    window_trx = df_transactions[(df_transactions["date"] >= window_start) & (df_transactions["date"] < snapshot)]

    # 1. Unique senders per receiver
    unique_senders = window_trx.groupby("RECEIVER_ACCOUNT_ID")[
        "SENDER_ACCOUNT_ID"
    ].nunique()

    # 2. Received volume concentration — % of total received amount from top sender
    amt_per_sender = window_trx.groupby(["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID"])[
        "TX_AMOUNT"
    ].sum()
    received_concentration = amt_per_sender.groupby(level=0).apply(
        lambda x: x.max() / x.sum()
    )

    # 3. Sender diversity — unique senders / total received transactions
    trx_received = window_trx.groupby("RECEIVER_ACCOUNT_ID")["TX_ID"].count()
    sender_diversity = unique_senders / trx_received

    # 4. Received to sent ratio
    total_received = window_trx.groupby("RECEIVER_ACCOUNT_ID")["TX_AMOUNT"].sum()
    total_sent = window_trx.groupby("SENDER_ACCOUNT_ID")["TX_AMOUNT"].sum()
    total_received.index.name = "ACCOUNT_ID"
    total_sent.index.name = "ACCOUNT_ID"
    recv_sent_ratio = total_received / total_sent.reindex(
        total_received.index, fill_value=1
    )

    # Combine — all naming happens here
    fanin = pd.DataFrame(
        {
            f"{prefix}_unique_senders": unique_senders,
            f"{prefix}_received_concentration": received_concentration,
            f"{prefix}_sender_diversity": sender_diversity,
            f"{prefix}_recv_sent_ratio": recv_sent_ratio,
        }
    )
    fanin.index.name = "ACCOUNT_ID"

    return fanin

# Test on April 1 snapshot, recent window (March)
test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

fanin_features = compute_fanin_features(df_transactions, test_snapshot, test_recent_start, "recent")
print(f"\nSnapshot: {test_snapshot.date()}")
print(f"\nWindow: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"\nAccounts with fan-in features: {len(fanin_features)}")
print(f"\nSample (first 5):")
print(fanin_features.head())



Snapshot: 2020-04-01

Window: 2020-03-01 to 2020-04-01

Accounts with fan-in features: 9778

Sample (first 5):
            recent_unique_senders  recent_received_concentration  \
ACCOUNT_ID                                                         
3                               1                            1.0   
5                               1                            1.0   
6                               1                            1.0   
7                               1                            1.0   
10                              1                            1.0   

            recent_sender_diversity  recent_recv_sent_ratio  
ACCOUNT_ID                                                   
3                          0.333333                0.162417  
5                          0.500000                0.117454  
6                          0.333333                0.803860  
7                          0.333333                0.648389  
10                         0.333333    

## 4.c Cycle Signals

Features that capture circular flow patters (money moving in loops between accounts). From the EDA, we found that in ~95% of cycle alerts, senders become receivers for the same alert id.

**Why recent window only (no baseline)?** Unlike intensity or fan-in, the presence of bidirectional relationships is the signal itself. An account with 10 bidirectional partners is suspicious whether that's an increase from 5 or has always been 10. The velocity features already capture behavioral change through intensity and fan-in metrics. Cycle velocity could be added as a future improvement.

Features:

- Sender-receiver-overlap: % of counterparties that are both senders to and receivers from this client
- bidirectional partners: Counts of accounts with both send and receive relationship

In [20]:
def compute_cycle_features(df_transactions, snapshot, window_start, prefix):

    """
    compute cycle features for all accounts wihin a given time window
    """

    # trx of interest

    window_trx = df_transactions[(df_transactions['date'] >= window_start) & (df_transactions['date']< snapshot)]

    # For each account find the account_ids the client is sending money to
    sent_to = window_trx.groupby('SENDER_ACCOUNT_ID')['RECEIVER_ACCOUNT_ID'].apply(set) 
    sent_to.index.name = "ACCOUNT_ID"

    # For each account find the account_ids they are receiving money from
    received_from = window_trx.groupby('RECEIVER_ACCOUNT_ID')['SENDER_ACCOUNT_ID'].apply(set)
    received_from.index.name = "ACCOUNT_ID"

    # To ensure sent to and received from have the same index (in case an account didn't receive/send money)
    all_accounts = sent_to.index.union(received_from.index)
    sent_to = sent_to.reindex(all_accounts, fill_value=set())
    received_from = sent_to.reindex(all_accounts, fill_value=set())

    # 1. Find bidirectional partners: # of accounts a client is both sending money and receiving money from
    bidirectional = sent_to.combine(received_from, lambda s,r:len(s&r))

    # 2. Overlap ratio: # cps with bidirectional relationship / # total counterparties
    total_counterparties = sent_to.combine(received_from, lambda s,r: (len(s|r)))
    overlap_ratio = bidirectional / total_counterparties.replace(0,1) # in case no cps - replace it with 1 to avoid err

    # Combine
    cycle = pd.DataFrame({

        f"{prefix}_bidirectional_partners": bidirectional,
        f"{prefix}_overlap_ratio":overlap_ratio})
    
    cycle.index.name = 'ACCOUNT_ID'

    return cycle

# Test on April 1 snapshot, recent window (March)
test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

cycle_features = compute_cycle_features(df_transactions, test_snapshot, test_recent_start, "recent")
print(f"Snapshot: {test_snapshot.date()}")
print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"Accounts with cycle features: {len(cycle_features)}")
print(f"\nSample (first 5):")
print(cycle_features.head())
print(f"\nAccounts with any bidirectional partners: {(cycle_features['recent_bidirectional_partners'] > 0).sum()}")    
print(cycle_features['recent_bidirectional_partners'].describe())


Snapshot: 2020-04-01
Window: 2020-03-01 to 2020-04-01
Accounts with cycle features: 9995

Sample (first 5):
            recent_bidirectional_partners  recent_overlap_ratio
ACCOUNT_ID                                                     
1                                       1                   1.0
2                                       1                   1.0
3                                       1                   1.0
4                                       1                   1.0
5                                       2                   1.0

Accounts with any bidirectional partners: 9818
count    9995.000000
mean        5.388594
std         6.604486
min         0.000000
25%         1.000000
50%         3.000000
75%         8.000000
max        48.000000
Name: recent_bidirectional_partners, dtype: float64


## 4.d Velocity/ Change

Velocity features compare recent (previous 1 month) to baseline (2 months before that) to detect behavioral shifts.
A sudden change in behavior is often more suspicious than absolute level of activity.

Features:

- `velocity_trx_sent` — recent sent count / baseline avg monthly sent count
- `velocity_trx_received` — recent received count / baseline avg monthly received count
- `velocity_amount_sent` — recent sent amount / baseline avg monthly sent amount
- `velocity_amount_received` — recent received amount / baseline avg monthly received amount
- `velocity_unique_senders` — recent unique senders / baseline avg unique senders
- `new_counterparty_ratio` — % of counterparties in recent window not seen in baseline

In [21]:
def compute_velocity_features(recent_intensity, baseline_intensity, recent_fanin, baseline_fanin):

    """
    Compute velocity features by comparing recent and baseline features

    Baseline is divided by 2 to get the avg before the comparison
    """

    idxx = recent_intensity.index
    velocity = pd.DataFrame(index = idxx)

    # 1. Baseline monthly averages 
    bl_avg_trx_sent = baseline_intensity['baseline_trx_sent'] / 2
    bl_avg_trx_received = baseline_intensity['baseline_trx_received'] / 2
    bl_avg_amt_sent = baseline_intensity['baseline_amt_sent'] / 2
    bl_avg_amt_received = baseline_intensity['baseline_amt_received'] / 2
    bl_avg_unique_senders = baseline_fanin['baseline_unique_senders']/2

    # 2. Compute Velocity Ratios ---- reindex to ensure same index, replace 0s with 1 to avoid division err
    velocity["velocity_trx_sent"] = recent_intensity["recent_trx_sent"] / bl_avg_trx_sent.reindex(idxx, fill_value=1).replace(0, 1)
    velocity["velocity_trx_received"] = recent_intensity["recent_trx_received"] / bl_avg_trx_received.reindex(idxx, fill_value=1).replace(0, 1)
    velocity["velocity_amt_sent"] = recent_intensity["recent_amt_sent"] / bl_avg_amt_sent.reindex(idxx, fill_value=1).replace(0, 1)
    velocity["velocity_amt_received"] = recent_intensity["recent_amt_received"] / bl_avg_amt_received.reindex(idxx, fill_value=1).replace(0, 1)
    velocity["velocity_unique_senders"] = recent_fanin["recent_unique_senders"] / bl_avg_unique_senders.reindex(idxx, fill_value=1).replace(0, 1)

    velocity.index.name = "ACCOUNT_ID"

    return velocity.fillna(0)



### Velocity Test ###
test_snapshot = snapshot_dates[0]
test_recent_start= test_snapshot - pd.DateOffset(months = 1)
test_baseline_start = test_snapshot - pd.DateOffset(months = 3)

# Compute recent and baseline features first
recent_intensity = compute_intensity_features(df_transactions, test_snapshot, test_recent_start, "recent")
baseline_intensity = compute_intensity_features(df_transactions, test_recent_start, test_baseline_start, "baseline")
recent_fanin = compute_fanin_features(df_transactions, test_snapshot, test_recent_start, "recent")
baseline_fanin = compute_fanin_features(df_transactions, test_recent_start, test_baseline_start, "baseline")

# Compute Velocity Features
velocity_features_test = compute_velocity_features(recent_intensity,baseline_intensity, recent_fanin, baseline_fanin)
print(f"Snapshot: {test_snapshot.date()}")
print(f"Recent: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"Baseline: {test_baseline_start.date()} to {test_recent_start.date()}")
print(f"Accounts: {len(velocity_features_test)}")
print(f"\nSample (first 5):")
print(velocity_features_test.head())


Snapshot: 2020-04-01
Recent: 2020-03-01 to 2020-04-01
Baseline: 2020-01-01 to 2020-03-01
Accounts: 9999

Sample (first 5):
            velocity_trx_sent  velocity_trx_received  velocity_amt_sent  \
ACCOUNT_ID                                                                
1                         2.0                    0.0                2.0   
2                         2.0                    0.0                2.0   
3                         2.0                    2.0                2.0   
4                         2.0                    0.0                2.0   
5                         2.0                    2.0                2.0   

            velocity_amt_received  velocity_unique_senders  
ACCOUNT_ID                                                  
1                             0.0                      0.0  
2                             0.0                      0.0  
3                             2.0                      2.0  
4                             0.0             

In [22]:
print("Account 1 - recent trx sent:", recent_intensity.loc[1, "recent_trx_sent"])
print("Account 1 - baseline trx sent:", baseline_intensity.loc[1, "baseline_trx_sent"])
print("Account 1 - baseline avg (÷2):", baseline_intensity.loc[1, "baseline_trx_sent"] / 2)


Account 1 - recent trx sent: 24
Account 1 - baseline trx sent: 24
Account 1 - baseline avg (÷2): 12.0


In [23]:
### Velocity Test ###
test_snapshot = snapshot_dates[0]
test_recent_start= test_snapshot - pd.DateOffset(months = 1)
test_baseline_start = test_snapshot - pd.DateOffset(months = 3)

print(test_snapshot, test_recent_start, test_baseline_start)


2020-04-01 00:00:00 2020-03-01 00:00:00 2020-01-01 00:00:00


## 4.e New Counterparty Ratio 

What % of counterparties in the recent window are new (not seen in the baseline window).A high ratio means client is suddenly transaction with lots of new people, which is suspicious for both fan-in and cycle patterns.

In [24]:
def compute_new_counterparty_ratio(df_transactions, snapshot, recent_start, baseline_start):

    """
    Compute % of counterparties in recent window, which were not seen in baseline window.
    """

    recent_trx = df_transactions[(df_transactions['date'] >= recent_start) & (df_transactions['date'] < snapshot)]
    baseline_trx = df_transactions[(df_transactions['date'] >= baseline_start) & (df_transactions['date'] < recent_start)]

    # get counterparties per account per window

    def get_counterparties(trx, account_col, counterparty_col):
        return trx.groupby(account_col)[counterparty_col].apply(set).rename_axis('ACCOUNT_ID')
    
    recent_sent = get_counterparties(recent_trx, 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID')
    recent_received = get_counterparties(recent_trx, 'RECEIVER_ACCOUNT_ID', 'SENDER_ACCOUNT_ID')
    baseline_sent = get_counterparties(baseline_trx, 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID')
    baseline_received = get_counterparties(baseline_trx, 'RECEIVER_ACCOUNT_ID', 'SENDER_ACCOUNT_ID')


    # all accounts with any trx made in the recent window (reindexing with recent window is sufficient)
    all_accounts = recent_sent.index.union(recent_received.index)


    # reindex sent and received to have to complete list of clientids in the set - if nothing sent/received, fill with set()
    recent_sent_filled = recent_sent.reindex(all_accounts, fill_value = set())
    recent_received_filled = recent_received.reindex(all_accounts, fill_value = set())
    recent_all = recent_sent_filled.combine(recent_received_filled, lambda s,r: s|r)
    

    baseline_sent_filled = baseline_sent.reindex(all_accounts, fill_value = set())
    baseline_received_filled = baseline_received.reindex(all_accounts, fill_value = set())
    baseline_all = baseline_sent_filled.combine(baseline_received_filled, lambda s,r: s|r)

    # New counterparty ratio

    new_ratio = recent_all.combine(baseline_all, lambda recent,base: len(recent-base)/len(recent) if len(recent)>0 else 0)

    return pd.DataFrame({"new_counterparty_ratio": new_ratio})


test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)
test_baseline_start = test_snapshot - pd.DateOffset(months=3)

new_cp = compute_new_counterparty_ratio(df_transactions, test_snapshot, test_recent_start, test_baseline_start)
print(f"Accounts: {len(new_cp)}")
print(f"\nSample:")
print(new_cp.head(10))
print(f"\nMean new counterparty ratio: {new_cp['new_counterparty_ratio'].mean():.2f}")
print(f"Max new counterparty ratio: {new_cp['new_counterparty_ratio'].max():.2f}")


Accounts: 9995

Sample:
            new_counterparty_ratio
ACCOUNT_ID                        
1                              0.0
2                              0.0
3                              0.0
4                              0.0
5                              0.0
6                              0.0
7                              0.0
8                              0.0
9                              0.0
10                             0.0

Mean new counterparty ratio: 0.09
Max new counterparty ratio: 1.00


# 5) Forward-Looking Labels

For each client at each snapshot date, the label answers: **will this client be involved in suspicious activity during the upcoming calendar month?**

- April 1 snapshot → label = 1 if client has alert in April
- May 1 snapshot → label = 1 if client has alert in May
- June 1 snapshot → label = 1 if client has alert in June

**Label logic:** An account is flagged (label = 1) only if BOTH conditions are met:
1. Involved in an alert transaction during the snapshot month (as sender or receiver)
2. Marked as a fraud account (`IS_FRAUD = 1`) in the accounts table

This dual condition ensures we only label accounts that are genuinely part of the laundering scheme.
The `IS_FRAUD` flag is our ground truth, regardless of whether the account acts as sender or receiver in the alert.

**Why alerts instead of IS_FRAUD alone?** The static `IS_FRAUD` flag doesn't tell us WHEN fraud happened. Using alerts gives us temporal labels, the same fraud account can be labeled 1 in April but 0 in May, letting the model learn what behavioral patterns PRECEDE suspicious activity. This is an approximation of True Positive labels for alerts.


**Observation:** In AMLSim, every account involved in an alert is also marked as `IS_FRAUD = 1`, there are zero false positives in the synthetic alert system. This means the dual condition (alerts ∩ IS_FRAUD) simplifies to just using alerts for this dataset. However, keeping the dual logic is good practice because in production AML systems, alerts frequently include FP accounts caught in the net.

In [25]:
def compute_labels(df_alerts, df_accounts, snapshot):
    """
    Label = 1 if account is fraud and appears in an alert during snapshot month.
    """
    month_end = snapshot + pd.offsets.MonthEnd(0)
    month_start = snapshot

    # Alerts in snapshot month
    window_alerts = df_alerts[
        (df_alerts["date"] >= month_start) & 
        (df_alerts["date"] <= month_end)
    ]
    
    # accounts involved in alerted transactions
    alerted_accounts = set(window_alerts['SENDER_ACCOUNT_ID']) | set(window_alerts['RECEIVER_ACCOUNT_ID'])

    # Only label accounts that are actually fraud (TP Appr.)
    fraudulent_accounts = set(df_accounts[df_accounts['IS_FRAUD'] == 1]['ACCOUNT_ID'])
    labeled_accounts = alerted_accounts & fraudulent_accounts


    print(f" On {snapshot.date()}, there were {len(window_alerts)} alerted transactions, {len(alerted_accounts)} accounts involved in an alerted trx and {len(labeled_accounts)} accounts with confirmed Fraud")
    return labeled_accounts

# Test on all snapshots
for snapshot in snapshot_dates:
    flagged = compute_labels(df_alerts, df_accounts, snapshot)


 On 2020-04-01, there were 200 alerted transactions, 235 accounts involved in an alerted trx and 235 accounts with confirmed Fraud
 On 2020-05-01, there were 246 alerted transactions, 292 accounts involved in an alerted trx and 292 accounts with confirmed Fraud
 On 2020-06-01, there were 293 alerted transactions, 342 accounts involved in an alerted trx and 342 accounts with confirmed Fraud


# 6) Final Dataset Construction

Loop through all 3 snapshots, compute all features and labels, and combine them into a single dataset.

For each snapshot:

1. Compute recent (snapshot - 1m) and baseline features (recent - 2m)
2. Compute velocity features: Comparing recent vs.baseline
3. Compute new counterparty ratio. 
4. Compute forward looking labels based on alerts
5. Join everything into a one row per client.

In [26]:
snapshot_dates


[Timestamp('2020-04-01 00:00:00'),
 Timestamp('2020-05-01 00:00:00'),
 Timestamp('2020-06-01 00:00:00')]

In [28]:
all_snapshots = []

for snapshot in snapshot_dates:
    print(f"\n{'*'*60}\nProcessing snapshot: {snapshot.date()}\n{'*'*60}")

    # define windows
    recent_start = snapshot - pd.DateOffset(months = 1)
    baseline_start = snapshot - pd.DateOffset(months = 3)

    print(f" Recent: {recent_start} to {snapshot.date()}")
    print(f" Baseline: {'baseline_start'} to {recent_start.date()}")


    # 1. Intensity features
    recent_intensity = compute_intensity_features(df_transactions, snapshot, recent_start, "recent")
    baseline_intensity = compute_intensity_features(df_transactions, recent_start, baseline_start, "baseline")

    # 2. Fan-in Features
    recent_fanin = compute_fanin_features(df_transactions, snapshot, recent_start, "recent")
    baseline_fanin = compute_fanin_features(df_transactions, recent_start, baseline_start, "baseline")

    # 3. Cycle Features - recent only 
    recent_cycle = compute_cycle_features(df_transactions, snapshot, recent_start, "recent")

    # 4. Velocity features
    velocity = compute_velocity_features(recent_intensity, baseline_intensity, recent_fanin, baseline_fanin)
    
    # 5. New counterparty ratio
    new_cp = compute_new_counterparty_ratio(df_transactions, snapshot, recent_start, baseline_start)
    
    # 6. Labels
    flagged_accounts = compute_labels(df_alerts, df_accounts, snapshot)


    # 7. Join all features
    snapshot_df = recent_intensity \
        .join(baseline_intensity, how="outer") \
        .join(recent_fanin, how="outer") \
        .join(baseline_fanin, how="outer") \
        .join(recent_cycle, how="outer") \
        .join(velocity, how="outer") \
        .join(new_cp, how="outer") \
        .fillna(0)
    

    # 8. Add metadata 
    snapshot_df['snapshot_date'] = snapshot
    snapshot_df['label'] = snapshot_df.index.isin(flagged_accounts).astype(int)

    print(f"Accounts: {len(snapshot_df)}")
    print(f" Number of accounts flagged: {snapshot_df['label'].sum()}")


    # combine dfs for each snapshot    
    all_snapshots.append(snapshot_df)

# combine list of dfs into a single df
modeling_df = pd.concat(all_snapshots).reset_index()
print(f"\n{'='*60}")
print(f"Final dataset: {modeling_df.shape[0]} rows × {modeling_df.shape[1]} columns")
print(f"Label distribution:\n{modeling_df['label'].value_counts()}")



************************************************************
Processing snapshot: 2020-04-01
************************************************************
 Recent: 2020-03-01 00:00:00 to 2020-04-01
 Baseline: baseline_start to 2020-03-01
 On 2020-04-01, there were 200 alerted transactions, 235 accounts involved in an alerted trx and 235 accounts with confirmed Fraud
Accounts: 9999
 Number of accounts flagged: 235

************************************************************
Processing snapshot: 2020-05-01
************************************************************
 Recent: 2020-04-01 00:00:00 to 2020-05-01
 Baseline: baseline_start to 2020-04-01
 On 2020-05-01, there were 246 alerted transactions, 292 accounts involved in an alerted trx and 292 accounts with confirmed Fraud
Accounts: 9999
 Number of accounts flagged: 292

************************************************************
Processing snapshot: 2020-06-01
************************************************************
 Recent: 20

In [29]:
# save the modeling df
output_path = Path("../data/processed")
output_path.mkdir(parents = True, exist_ok = True)

modeling_df.to_csv(output_path / "modeling_dataset.csv", index=False)

print(modeling_df.shape)


(29997, 31)


# 7) Dataset Validation


Before starting with the modeling, let's verify that dataset is clean and free of data leakage.

In [32]:
# 1. Nulls and infinites

print("*** Null Check ***")

null_counts = modeling_df.isnull().sum()
if null_counts.sum() == 0:
    print('\nNo null values')
else:
    print('\nNulls found')
    print(null_counts[null_counts]>0)

print("\n*** Infinity Check ***")
numeric_cols = modeling_df.select_dtypes(include = [np.number]).columns
inf_counts = np.isinf(modeling_df[numeric_cols]).sum()
if inf_counts.sum() == 0:
    print("\n✅ No infinite values")
else:
    print("❌ Infinities found:")
    print(inf_counts[inf_counts > 0])


# 2. Class balance per snapshot
print("\n*** Class balance ***")
for snapshot in snapshot_dates:
    snapshot_data = modeling_df[modeling_df['snapshot_date'] == snapshot]
    fraud_count = snapshot_data['label'].sum()
    total = snapshot_data.shape[0]
    print(f"  {snapshot.date()} → {fraud_count} fraud / {total} total ({fraud_count/total*100:.2f}%)")

# 3. Feature Summary


print("\n*** Infinity Check ***")

feature_cols = [c for c in modeling_df.columns if c not in ["ACCOUNT_ID", "snapshot_date", "label"]]
print(f"/nTotal features: {len(feature_cols)}")
print(f"\nFeature Stats: ")
print()
print(modeling_df[feature_cols].describe().T[["mean", "std", "min", "max"]].to_string())


*** Null Check ***

No null values

*** Infinity Check ***

✅ No infinite values

*** Class balance ***
  2020-04-01 → 235 fraud / 9999 total (2.35%)
  2020-05-01 → 292 fraud / 9999 total (2.92%)
  2020-06-01 → 342 fraud / 9999 total (3.42%)

*** Infinity Check ***
/nTotal features: 28

Feature Stats: 

                                         mean           std     min           max
recent_trx_sent                  1.323366e+02  1.327378e+02    7.00  9.630000e+02
recent_amt_sent                  1.534949e+07  3.562090e+07  987.21  2.357803e+08
recent_avg_amt_sent              7.618895e+05  1.743592e+06    4.63  1.121475e+07
recent_trx_received              1.323366e+02  1.944203e+02    0.00  4.843000e+03
recent_amt_received              1.534949e+07  2.388747e+07    0.00  4.894107e+08
recent_avg_amt_received          2.651143e+05  8.887292e+05    0.00  2.147484e+07
baseline_trx_sent                1.323366e+02  1.327378e+02    7.00  9.630000e+02
baseline_amt_sent                1.5349

In [34]:
print(modeling_df[feature_cols].describe())#.T[["mean", "std", "min", "max"]].to_string())


       recent_trx_sent  recent_amt_sent  recent_avg_amt_sent  \
count     29997.000000     2.999700e+04         2.999700e+04   
mean        132.336634     1.534949e+07         7.618895e+05   
std         132.737822     3.562090e+07         1.743592e+06   
min           7.000000     9.872100e+02         4.630000e+00   
25%          22.000000     4.924920e+03         1.263800e+02   
50%         132.000000     2.872760e+04         2.964800e+02   
75%         180.000000     1.006362e+05         5.498800e+02   
max         963.000000     2.357803e+08         1.121475e+07   

       recent_trx_received  recent_amt_received  recent_avg_amt_received  \
count         29997.000000         2.999700e+04             2.999700e+04   
mean            132.336634         1.534949e+07             2.651143e+05   
std             194.420333         2.388747e+07             8.887292e+05   
min               0.000000         0.000000e+00             0.000000e+00   
25%              45.000000         1.911367

In [40]:
print(modeling_df[feature_cols].describe())


       recent_trx_sent  recent_amt_sent  recent_avg_amt_sent  \
count     29997.000000     2.999700e+04         2.999700e+04   
mean        132.336634     1.534949e+07         7.618895e+05   
std         132.737822     3.562090e+07         1.743592e+06   
min           7.000000     9.872100e+02         4.630000e+00   
25%          22.000000     4.924920e+03         1.263800e+02   
50%         132.000000     2.872760e+04         2.964800e+02   
75%         180.000000     1.006362e+05         5.498800e+02   
max         963.000000     2.357803e+08         1.121475e+07   

       recent_trx_received  recent_amt_received  recent_avg_amt_received  \
count         29997.000000         2.999700e+04             2.999700e+04   
mean            132.336634         1.534949e+07             2.651143e+05   
std             194.420333         2.388747e+07             8.887292e+05   
min               0.000000         0.000000e+00             0.000000e+00   
25%              45.000000         1.911367

# 8) Summary and Next Steps

## Dataset Overview

- 29,997 rows x 31 columns (9999 accounts x 3 months snapshots)

- Features: 28 Behavioral Features Across 5 Categories (intensity, fan-in, cycle, velocity, new counterparty)

- Label: ~3.4% Positive rate (Alerts in snapshot months out of fraudulent accounts)



## Feature Categories
| Category | Count | Description |
|---|---|---|
| Intensity (recent) | 6 | Transaction counts, amounts, averages (sent/received) |
| Intensity (baseline) | 6 | Same metrics for baseline period |
| Fan-in (recent) | 4 | Unique senders, concentration, diversity, recv/sent ratio |
| Fan-in (baseline) | 4 | Same metrics for baseline period |
| Cycle | 2 | Bidirectional partners, overlap ratio |
| Velocity | 5 | Recent vs baseline ratios |
| New counterparty | 1 | % of counterparties not seen in baseline |


## Observations

- HOW?? Zero variance features: 'velocity_trx_sent' and 'velocity_amt_sent' have std = 0 (all values are 2). These carry no information. Drop before modeling.

- HOW and WHY?? - `recv_sent_ratio` has extreme outliers (max ~16.8M). May need capping or log-transform for Logistic Regression.

- HOW: Class imbalance: ~3.4% positive rate is much healthier than the 0.13% at transaction level, but still imbalanced. Will need class weights or threshold tuning.

- AMLSim Data observation: All accounts involved in alerts are also IS_FRAUD=1 (zero false positives in synthetic data). In real world scenario, this wouldn't be the case.


## Next Steps:

## Next Steps
→ See `03_modeling.ipynb`:
1. **Drop zero-variance features** and handle extreme values
2. **Time-based train/test split:** April + May snapshots for training, June for testing
3. **Rule-based baseline:** Simple threshold rules on fan-in features
4. **Logistic Regression:** Interpretable risk scoring model
5. **LightGBM:** Non-linear model to capture feature interactions
6. **Evaluation:** Precision@TopK, recall within alert budget, SHAP explainability